<a href="https://colab.research.google.com/github/SeiDra/lending-club-prediction/blob/scindage-du-notebook-FeatureEngineering%2FModeling/du_sda_ml2_P4_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROJET 7 : Loan Default Prediction
Partie N°4 — Modélisation & Évaluation

Contenu :
- Import des données transformées (sortie du P3)
- Gestion du déséquilibre (Undersampling vs SMOTE)
- Feature Selection
- Comparaison de modèles (Cross-Validation 5-fold)
- Optimisation du meilleur modèle
- Évaluation finale sur le Test Set (une seule fois)
- Interprétabilité et conclusions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as sps
import seaborn as sns

from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score, precision_score,
                              recall_score, roc_auc_score, roc_curve,
                              precision_recall_curve)
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

In [ ]:
train_X = pd.read_parquet("DATA/03_train_X.parquet")
train_y = pd.read_parquet("DATA/03_train_y.parquet").squeeze()
test_X = pd.read_parquet("DATA/03_test_X.parquet")
test_y = pd.read_parquet("DATA/03_test_y.parquet").squeeze()

# Lecture du nom de la cible (pour les fonctions utilitaires)
with open("CONFIG/target_config.txt", "r") as f:
    target_col = f.read().strip()

print(f"Train X : {train_X.shape} | Train y : {train_y.shape}")
print(f"Test  X : {test_X.shape}  | Test  y : {test_y.shape}")
print(f"\nRatio défaut train : {train_y.mean():.2%}")
print(f"Ratio défaut test  : {test_y.mean():.2%}")

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# Option A : Undersampling
undersampler = RandomUnderSampler(random_state=42)
train_X_under, train_y_under = undersampler.fit_resample(train_X, train_y)
print(f"Undersampling : {train_X_under.shape}")
print(train_y_under.value_counts())

# Option B : SMOTE
smote = SMOTE(random_state=42)
train_X_smote, train_y_smote = smote.fit_resample(train_X, train_y)
print(f"\nSMOTE : {train_X_smote.shape}")
print(train_y_smote.value_counts())

In [ ]:
# Méthode : Feature Importance via un modèle LightGBM rapide
from lightgbm import LGBMClassifier

# Entraîner un modèle rapide pour obtenir les importances
lgbm_quick = LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
lgbm_quick.fit(train_X_under, train_y_under)

# Extraire et afficher les importances
importances = pd.Series(lgbm_quick.feature_importances_, index=train_X_under.columns)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importances.head(20).plot(kind='barh')
plt.title("Top 20 Features par Importance (LightGBM)")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Sélectionner les features avec importance > 0 (ou un seuil personnalisé)
# ou choisir un nombre fixe (ex: top 15)
N_TOP_FEATURES = 15
vars_final = importances.head(N_TOP_FEATURES).index.tolist()
print(f"\n{N_TOP_FEATURES} features retenues : {vars_final}")

# Filtrer train et test
train_X_under_selected = train_X_under[vars_final]
train_X_smote_selected = train_X_smote[vars_final]
test_X_selected = test_X[vars_final]

In [ ]:
def calculate_scores(model, X_trn, y_trn, X_tst, y_tst):
    """Entraîne le modèle et calcule les métriques sur le jeu de test."""
    model.fit(X_trn, y_trn)
    y_pred = model.predict(X_tst)
    y_pred_proba = model.predict_proba(X_tst)[:, 1]

    accuracy = accuracy_score(y_tst, y_pred)
    conf_matrix = confusion_matrix(y_tst, y_pred)
    precision = precision_score(y_tst, y_pred)
    recall = recall_score(y_tst, y_pred)
    f1 = f1_score(y_tst, y_pred)
    auc = roc_auc_score(y_tst, y_pred_proba)

    # KS Statistic
    mask = y_tst.astype(bool).values
    ks = sps.ks_2samp(y_pred_proba[mask], y_pred_proba[~mask])[0]

    return accuracy, auc, ks, conf_matrix, precision, recall, f1


def calculate_cv_scores(model, X, y, cv=5):
    """Cross-validation stratifiée 5-fold avec métriques complètes."""
    kf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    results = {'accuracy': [], 'auc': [], 'ks': [],
               'precision': [], 'recall': [], 'f1': [], 'conf_matrices': []}

    for train_idx, test_idx in kf.split(X, y):
        X_trn, X_tst = X.iloc[train_idx], X.iloc[test_idx]
        y_trn, y_tst = y.iloc[train_idx], y.iloc[test_idx]

        acc, auc, ks, cm, prec, rec, f1 = calculate_scores(
            model, X_trn, y_trn, X_tst, y_tst)

        results['accuracy'].append(acc)
        results['auc'].append(auc)
        results['ks'].append(ks)
        results['conf_matrices'].append(cm)
        results['precision'].append(prec)
        results['recall'].append(rec)
        results['f1'].append(f1)

    return {k: np.mean(v) for k, v in results.items() if k != 'conf_matrices'}, \
           np.mean(results['conf_matrices'], axis=0)


def compare_models(models, X, y, method_name="", cv=5):
    """Compare plusieurs modèles avec CV et retourne un tableau récapitulatif."""
    summary = pd.DataFrame(columns=['accuracy', 'auc', 'ks', 'precision', 'recall', 'f1'])

    for name, model in models.items():
        print(f"  ⏳ {name}...", end="")
        scores, mean_cm = calculate_cv_scores(model, X, y, cv=cv)
        label = f"{name} ({method_name})" if method_name else name
        summary.loc[label] = [scores['accuracy'], scores['auc'], scores['ks'],
                              scores['precision'], scores['recall'], scores['f1']]
        print(f" ✅ AUC={scores['auc']:.4f} | F1={scores['f1']:.4f}")

    return summary

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(min_samples_split=60, min_samples_leaf=30),
    'KNN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100, min_samples_split=60,
                                            min_samples_leaf=30, random_state=42),
    'Naive Bayes': GaussianNB(),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42,
                              use_label_encoder=False, eval_metric='logloss'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
}

# Comparaison avec Undersampling
print("=" * 60)
print("COMPARAISON DES MODÈLES — UNDERSAMPLING")
print("=" * 60)
results_under = compare_models(models, train_X_under_selected, train_y_under,
                                method_name="Undersample")

# Comparaison avec SMOTE
print("\n" + "=" * 60)
print("COMPARAISON DES MODÈLES — SMOTE")
print("=" * 60)
results_smote = compare_models(models, train_X_smote_selected, train_y_smote,
                                method_name="SMOTE")

# Tableau comparatif final
all_results = pd.concat([results_under, results_smote])
all_results = all_results.sort_values('auc', ascending=False)

print("\n" + "=" * 60)
print("TABLEAU COMPARATIF COMPLET")
print("=" * 60)
print(all_results.to_string())

In [ ]:
# On sélectionne le meilleur modèle d'après les résultats de la section précédente
# pour l'optimiser
# Exemple avec LightGBM (qu'on adaptera selon le meilleur modèle)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10, -1],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [15, 31, 63, 127],
    'min_child_samples': [10, 20, 50],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
}

search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    param_distributions=param_dist,
    n_iter=30,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Entraîner sur le jeu undersamplé (ou SMOTE selon les résultats)
search.fit(train_X_under_selected, train_y_under)

print(f"\nMeilleurs paramètres : {search.best_params_}")
print(f"Meilleur AUC (CV) : {search.best_score_:.4f}")

best_model = search.best_estimator_

In [ ]:
# On établit l'évaluation finale. On n'utilise le test set qu'ici.

y_pred = best_model.predict(test_X_selected)
y_pred_proba = best_model.predict_proba(test_X_selected)[:, 1]

# Classification Report
print("=" * 60)
print("RAPPORT DE CLASSIFICATION FINAL (TEST SET)")
print("=" * 60)
print(classification_report(test_y, y_pred, target_names=['Good Loan', 'Bad Loan']))

# Matrice de confusion
cm = confusion_matrix(test_y, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Good Loan', 'Bad Loan'],
            yticklabels=['Good Loan', 'Bad Loan'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Matrice de Confusion — Test Set')
plt.tight_layout()
plt.show()

# Courbe ROC
fpr, tpr, _ = roc_curve(test_y, y_pred_proba)
auc_score = roc_auc_score(test_y, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc_score:.4f}')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Courbe ROC — Test Set')
plt.legend()
plt.tight_layout()
plt.show()

# Courbe Precision-Recall
precision_curve, recall_curve, _ = precision_recall_curve(test_y, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(recall_curve, precision_curve)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Courbe Precision-Recall — Test Set')
plt.tight_layout()
plt.show()

# KS Statistic
mask = test_y.astype(bool).values
ks_stat = sps.ks_2samp(y_pred_proba[mask], y_pred_proba[~mask])[0]
print(f"\nKS Statistic : {ks_stat:.4f}")
print(f"AUC-ROC      : {auc_score:.4f}")

In [ ]:
# Feature Importance du modèle final
importances_final = pd.Series(best_model.feature_importances_,
                               index=test_X_selected.columns)
importances_final = importances_final.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances_final.plot(kind='barh', color='steelblue')
plt.title("Feature Importance — Modèle Final")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# (Optionnel) SHAP Values
# import shap
# explainer = shap.TreeExplainer(best_model)
# shap_values = explainer.shap_values(test_X_selected)
# shap.summary_plot(shap_values[1], test_X_selected)

In [ ]:
# Conclusions et Recommandations
# Résumer :
# 1. Les performances du modèle final (AUC, F1, KS, etc.)
# 2. Les 5 variables les plus influentes et leur interprétation métier
# 3. Les recommandations concrètes pour l'institution financière
# 4. Les limites du modèle et les pistes d'amélioration